# Unit 9 practical: evaluate a RAG application with a real model and a model judge
> **Status: written but NOT executed** while the course was authored (no GPU or model there). The model exists on the Hugging Face hub (checked).

You will: (1) generate answers to the course's 30 questions with a small chat model from retrieved context, (2) score them with the course's metrics, (3) ask the **model itself to judge groundedness in both orders** to expose position bias, (4) compare the judge with **your own labels** using Cohen's kappa, and (5) write the report with intervals.

In [ ]:
!pip -q install transformers accelerate
!wget -q https://krunalpanchal1995.github.io/banu.github.io/downloads/u9/u9-code.zip -O u9.zip && unzip -q -o u9.zip
import sys; sys.path.insert(0, "u9-code/codes/evaluation/evaluation_np"); sys.path.insert(0, "u9-code/codes/rag/rag_np")
import evaluation_np as E, rag_np as R, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
name = "Qwen/Qwen2.5-0.5B-Instruct"; tok = AutoTokenizer.from_pretrained(name); model = AutoModelForCausalLM.from_pretrained(name, torch_dtype="auto", device_map="auto")
def chat(system, user, n=120):
    ids = tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}], add_generation_prompt=True, return_tensors="pt").to(model.device)
    return tok.decode(model.generate(ids, max_new_tokens=n, do_sample=False)[0, ids.shape[1]:], skip_special_tokens=True).strip()

## 1. Generate answers from retrieved context

In [ ]:
corpus = R.load_corpus(); full = " ".join(corpus.values()); Q = R.load_queries()["answerable"]
chunks = R.chunk_corpus(corpus, 600, 120); T = [c["text"] for c in chunks]; bm = R.BM25(T)
rows = []
for q in Q:
    order = R.rank(bm.scores(q["q"])); ctx = " ".join(T[int(i)] for i in order[:3]); ref = E.sentence_with(full, q["gold"])
    ans = chat("Answer ONLY from the passages. If they do not contain the answer, say you do not know. One or two sentences.", f"Passages: {ctx}\n\nQuestion: {q['q']}")
    rows.append({"q": q["q"], "ctx": ctx, "ref": ref, "ans": ans})
print(rows[0]["ans"])

## 2. Automatic metrics (and their limits, Figure 1)

In [ ]:
f1 = np.array([E.token_f1(r["ref"] or "", r["ans"]) for r in rows]); rl = np.array([E.rouge_l(r["ref"] or "", r["ans"])[2] for r in rows])
gr = np.array([E.grounded_score(r["ans"], r["ctx"]) for r in rows])
print("token F1", f1.mean().round(3), "ROUGE-L", rl.mean().round(3), "groundedness (lexical + numbers)", gr.mean().round(3))

## 3. A model judge, in both orders (Section 3.3.1)
Ask which of two answers is better supported by the passage. The two answers are your model's answer and the reference sentence. Show them in both orders and look at the position bias.

In [ ]:
def judge(ctx, a, b):
    out = chat("You are a strict grader. Reply with exactly 'A' or 'B'.", f"Passage: {ctx}\n\nAnswer A: {a}\nAnswer B: {b}\n\nWhich answer is better supported by the passage?", n=3)
    return "A" if out.strip().upper().startswith("A") else "B"
first = [judge(r["ctx"], r["ans"], r["ref"]) == "A" for r in rows]      # model answer shown first: judge picks it?
second = [judge(r["ctx"], r["ref"], r["ans"]) == "B" for r in rows]     # model answer shown second
print("picks the model answer when it is first:", np.mean(first).round(2), " when it is second:", np.mean(second).round(2))
print("A large difference between these two numbers is position bias (compare Figure 2).")

## 4. Agreement with your own labels (kappa, Section 3.3.2)
Label 20 answers yourself (1 = supported by the passage, 0 = not), then compare with the model judge's verdicts (is the answer supported?).

In [ ]:
mine = [None] * 20          # TODO: fill in 20 labels of 0/1 after reading rows[0..19]
judge_says = [int(judge(r["ctx"], r["ans"], "I do not know.") == "A") for r in rows[:20]]
if all(m is not None for m in mine):
    print("raw agreement", np.mean(np.array(mine) == np.array(judge_says)).round(2), " Cohen's kappa", round(E.cohen_kappa(mine, judge_says), 2))

## 5. The report
Write a table like Figure 6 of the PDF: each metric with a 95% interval (`E.wilson` for rates, bootstrap for means) and one paragraph saying which conclusions the 30 questions can and cannot support.

## Your turn
1. Add 10 near-miss unanswerable questions (Unit 5). Does the model say "I do not know" or invent an answer? Measure the invention rate with `E.grounded_score`.
2. Change a number in three answers and check that `numbers_supported` catches it.
3. Run the fairness-style audit on *language*: ask the same question in two phrasings and compare success rates with Wilson intervals.